# Chapter 26 — Discovery Is Not Promotion

**Companion to *Applied AI*.**

A stance-diverse portfolio loses to plain redraws — and then, inside the
loss, one wording looks promising. This notebook recomputes the frozen P2
run (288 calls, 12 tasks) from its rows, recovers the stance each row never
recorded, and shows why the attractive subgroup was *not* promoted.

## Question

**What additional evidence would be required before this discovered pattern
deserves promotion?**

## What this notebook does

It **reproduces** `p-series/p2/p2-export.json`: the preregistered loss
(9/12 against 10/12 at ×1.22 tokens, drop rule fired), the stance recovery
by input-token signature (0 / +32 / +37 / +44, 36 calls each), per-stance
solve sets 7 / 7 / 7 / 9 — and the exposure-mismatch trap that keeps the
counterfactual signal fenced until a matched replication.

```text
post-hoc discovery  ≠  validated promotion
```

## Setup

Standard library only. No network, no API key, no `codeai` import. Only
bundle-relative paths are shown; override the evidence root with
`APPLIED_AI_EVIDENCE`.

In [1]:
import json
import os
from collections import Counter, defaultdict
from pathlib import Path

def find_evidence_dir(marker="p-series"):
    """Locate the preserved evidence. Override with APPLIED_AI_EVIDENCE."""
    env = os.environ.get("APPLIED_AI_EVIDENCE")
    if env and Path(env).expanduser().is_dir():
        return Path(env).expanduser()
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base / "evidence",
                     base / "experiments" / "applied-ai" / "evidence"):
            if (cand / marker).is_dir():
                return cand
    raise FileNotFoundError(
        "Preserved evidence not found. Set APPLIED_AI_EVIDENCE to the "
        "directory holding the Applied AI evidence bundles.")

EVIDENCE_DIR = find_evidence_dir()
EXPORT = EVIDENCE_DIR / "p-series" / "p2" / "p2-export.json"
print("bundle: p-series/p2/p2-export.json")
data = json.loads(EXPORT.read_text(encoding="utf-8"))
cands = data["candidates"]
calls = {c["call_id"]: c for c in data["calls"]}
print("arms: P2C = normal x12 | P2S = stance portfolio (4 wordings x3)")
print("candidates:", len(cands))

bundle: p-series/p2/p2-export.json
arms: P2C = normal x12 | P2S = stance portfolio (4 wordings x3)
candidates: 288


## 1. The preregistered result first: the portfolio lost

H1 of the hypothesis: stance-diverse sampling beats same-count normal
sampling. Recomputed from the rows — normal covers 10/12, the portfolio 9/12,
at ×1.22 tokens, with *more* candidate passes (53 vs 51) and fewer tasks.
The drop rule fired; the idea was dropped.

In [2]:
PASS = "VERIFIER_PASS"
solved = defaultdict(set)
passes = defaultdict(int)
toks = defaultdict(int)
for c in cands:
    call = calls[c["call_id"]]
    toks[c["arm"]] += call["input_tokens"] + call["output_tokens"]
    if c["outcome"] == PASS:
        solved[c["arm"]].add(c["corpus_task_id"])
        passes[c["arm"]] += 1

print(f"P2C normal   : {len(solved['P2C'])}/12 tasks, {passes['P2C']} candidate passes")
print(f"P2S portfolio: {len(solved['P2S'])}/12 tasks, {passes['P2S']} candidate passes")
print(f"token ratio P2S/P2C: {toks['P2S'] / toks['P2C']:.2f}x")
assert len(solved["P2C"]) == 10 and len(solved["P2S"]) == 9
assert (passes["P2C"], passes["P2S"]) == (51, 53)
print()
print("More passes, fewer tasks: twelve draws of one wording covered more")
print("ground than three draws each of four. The portfolio lost — on the")
print("books, before any subgroup is admired.")

P2C normal   : 10/12 tasks, 51 candidate passes
P2S portfolio: 9/12 tasks, 53 candidate passes
token ratio P2S/P2C: 1.22x

More passes, fewer tasks: twelve draws of one wording covered more
ground than three draws each of four. The portfolio lost — on the
books, before any subgroup is admired.


## 2. Recovering the variable the rows never recorded

The frozen export carries no stance field: both arms export identical
configurations, every call the same prompt version — though the ledger
recorded each call's variant and the exporter never copied it out. But every
normal-arm call on a task used the same input-token count, so subtracting
that baseline from each portfolio call on the same task leaves exactly four
values, 36 calls each. Recover the manipulated variable:

In [3]:
base = {}
for c in cands:
    if c["arm"] == "P2C":
        base[c["corpus_task_id"]] = calls[c["call_id"]]["input_tokens"]

deltas = Counter()
for c in cands:
    if c["arm"] == "P2S":
        deltas[calls[c["call_id"]]["input_tokens"] - base[c["corpus_task_id"]]] += 1
print("observed deltas:", dict(sorted(deltas.items())))
assert deltas == {0: 36, 32: 36, 37: 36, 44: 36}

# Zero delta can only be the empty normal suffix; the rest order with the
# frozen suffix lengths (185, 220, 290 chars); +44 is cross-checked by P3,
# where counterfactual is a whole arm at exactly +44 over normal.
STANCE = {0: "normal", 32: "minimality", 37: "assumption_challenge",
          44: "counterfactual"}
print("stance assignment:", STANCE)
print()
print("An inference, labeled as one: assumption-challenge vs minimality rests")
print("on length order alone. A signature that reproduces a report is strong;")
print("it is still an inference.")

observed deltas: {0: 36, 32: 36, 37: 36, 44: 36}
stance assignment: {0: 'normal', 32: 'minimality', 37: 'assumption_challenge', 44: 'counterfactual'}

An inference, labeled as one: assumption-challenge vs minimality rests
on length order alone. A signature that reproduces a report is strong;
it is still an inference.


## 3. Per-stance figures, recomputed — not quoted

Attributed by signature, the rows reproduce the report's per-stance figures
exactly: candidate passes 11 / 11 / 15 / 16 of 36, solve sets of 7 / 7 / 7
and 9 tasks:

In [4]:
stance_pass = Counter()
stance_solved = defaultdict(set)
for c in cands:
    if c["arm"] == "P2S":
        st = STANCE[calls[c["call_id"]]["input_tokens"] - base[c["corpus_task_id"]]]
        if c["outcome"] == PASS:
            stance_pass[st] += 1
            stance_solved[st].add(c["corpus_task_id"])

order = ["normal", "assumption_challenge", "minimality", "counterfactual"]
for st in order:
    print(f"{st:<20}passes {stance_pass[st]:>2}/36   solves {len(stance_solved[st])}/12")
assert [stance_pass[s] for s in order] == [11, 11, 15, 16]
assert [len(stance_solved[s]) for s in order] == [7, 7, 7, 9]
print()
print("Counterfactual's 9 contain normal's 7, plus stable-priority and")
print("strip-query. That is the attractive subgroup. Now fence it.")

normal              passes 11/36   solves 7/12
assumption_challengepasses 11/36   solves 7/12
minimality          passes 15/36   solves 7/12
counterfactual      passes 16/36   solves 9/12

Counterfactual's 9 contain normal's 7, plus stable-priority and
strip-query. That is the attractive subgroup. Now fence it.


## 4. The exposure-mismatch trap

At matched three-draw exposure, counterfactual covers 9 tasks against
normal's 7 — a lead. But both of its extra tasks are tasks the normal wording
*also* solved somewhere among its twelve draws. The comparison that produced
the lead gave one side four times the exposure:

In [5]:
extra = sorted(stance_solved["counterfactual"] - stance_solved["normal"])
print("counterfactual-only tasks:", extra)
assert extra == ["v2-stable-priority", "v2-strip-query"]
assert all(t in solved["P2C"] for t in extra)
print("also solved by normal x12:", True)
print()
print("The signal earns a matched replication — twelve draws per task each,")
print("same tasks, one variable changed — and nothing more. Promoting it now")
print("would credit the wording for what exposure explains. That replication")
print("is Chapter 27.")

counterfactual-only tasks: ['v2-stable-priority', 'v2-strip-query']
also solved by normal x12: True

The signal earns a matched replication — twelve draws per task each,
same tasks, one variable changed — and nothing more. Promoting it now
would credit the wording for what exposure explains. That replication
is Chapter 27.


## Interpretation

1. **The portfolio lost first.** 9/12 vs 10/12 at ×1.22 tokens; the drop
   rule fired. Discovery happens inside a recorded loss, not instead of it.
2. **Lost variables can sometimes be recovered.** The token signature
   reproduces the report exactly — and stays labeled an inference, with its
   three legs (zero-delta, length order, P3 cross-check) named.
3. **Matched exposure or it didn't happen.** A 9-vs-7 lead at mismatched
   exposure is a signal fenced for replication, never a promotion.
4. **What promotion would require:** the matched P3 replication to confirm,
   under a frozen rule, with the exposure confound removed.

## Try it yourself

1. Recompute H2 (decorrelation) from the rows: normal, assumption-challenge
   and minimality solve exactly the same seven tasks. What does that say
   about wording variety decorrelating failures?
2. The exporter dropped the stance field (repaired in current source, frozen
   export unchanged). Find the regression test that prevents a repeat —
   what does it assert about exported call rows?
3. Invent a promotion rule that the counterfactual subgroup *would* pass
   today. Then break it: what future evidence could embarrass that rule?

*Evidence: `experiments/applied-ai/evidence/p-series/p2/p2-export.json`
(288 calls, 288 candidates, byte-pinned export). No network, no API key, no
`codeai` import.*